In [118]:
##2.2 File tokenization, breaking  raw text into split tokens , import text file

with open("the-verdict.txt",'r', encoding = "utf8") as f:
    raw_text = f.read()
print(len(raw_text))

20398


In [119]:
##2.3 use split words from 2.2|1 and assign them unique token ids for vocab
import re

preprocessed = re.split(r'([,.?_!"()\']|--|\s)',raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
##print(preprocessed[:10])

##sort all words
all_words = sorted(set(preprocessed))
print(len(all_words))

##20398 -> 1162

##make vocab from all words

vocab = {}
for integer,words in enumerate(all_words):
    vocab[words] = integer
for i,item in enumerate(vocab.items()):
    print(item,'\n')
    if i == 10:
        break

1159
('!', 0) 

('"', 1) 

("'", 2) 

('(', 3) 

(')', 4) 

(',', 5) 

('--', 6) 

('.', 7) 

(':', 8) 

(';', 9) 

('?', 10) 



In [120]:
##2.3|2 use the vocab to createa tokenizer class that can encoder and decoder functions

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.string_to_int = vocab
        self.int_to_string = {i:s for s,i in vocab.items()}
    def encode(self,text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)',text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [ self.string_to_int[s] for s in preprocessed]
        return ids
    def decode(self,ids):
        text = " ".join([self.int_to_string[id] for id in ids])
        text =  re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [121]:
tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode(""""It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride.""")
text = tokenizer.decode(ids)
print(ids)
print(text)

[1, 58, 2, 872, 1013, 615, 541, 763, 5, 1155, 608, 5, 1, 69, 7, 39, 873, 1136, 773, 812, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [122]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab))
print(all_words[-10:])
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1161
['would', 'wouldn', 'year', 'years', 'yellow', 'yet', 'you', 'younger', 'your', 'yourself']
('younger', 1156)
('your', 1157)
('yourself', 1158)
('<|endoftext|>', 1159)
('<|unk|>', 1160)


In [123]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.string_to_int = vocab
        self.int_to_string = {i:s for s,i in vocab.items()}
    def encode(self,text):
                 
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)',text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item  if item in self.string_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [ self.string_to_int[s] for s in preprocessed]
        return ids
    def decode(self,ids):
        text = " ".join([self.int_to_string[id] for id in ids])
        text =  re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [124]:
tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode("Hello love, <|endoftext|> tea, park, peach")
decoded_ids = tokenizer.decode(ids)
print(ids)
print(decoded_ids)

[1160, 1160, 5, 1159, 1000, 5, 1160, 5, 1160]
<|unk|> <|unk|>, <|endoftext|> tea, <|unk|>, <|unk|>


In [125]:


#Byte Pair Coding
from importlib.metadata import version


import tiktoken

print(version("tiktoken"))

0.13.0


In [126]:
tokenizer = tiktoken.get_encoding("gpt2") 
type(tiktoken.list_encoding_names())
print(tiktoken.list_encoding_names())

['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']


In [127]:

#tokenizer.decode?


In [128]:
ids2 = tokenizer.encode("Hello love, <|endoftext|> tea, park, peach", allowed_special = "all")
decoded_ids2 = tokenizer.decode(ids2)
print(ids2)
print(decoded_ids2)

[15496, 1842, 11, 220, 50256, 8887, 11, 3952, 11, 47565]
Hello love, <|endoftext|> tea, park, peach


In [129]:
#Data Sampling 2.6

import torch

In [130]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5064


In [131]:
enc_sample = enc_text[50:]

In [132]:

context_size = 4         #1
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [133]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [134]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)    #1

        for i in range(0, len(token_ids) - max_length, stride):     #2
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):    #3
        return len(self.input_ids)

    def __getitem__(self, idx):         #4
        return self.input_ids[idx], self.target_ids[idx]

In [135]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")                         #1
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)   #2
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,     #3
        num_workers=num_workers     #4
    )

    return dataloader

In [136]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)      #1
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [137]:
#2.7
#App A

import torch
torch.cuda.is_available()


False

In [138]:
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [139]:
input_ids =  torch.tensor([2,3,5,2])
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(6,3)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [140]:
 embedding_layer(input_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)

In [151]:
#embedding layer for BPE tokenizer
vocab_size = 50257
output_dim = 256
max_length = 4
token_emebedding_layer = torch.nn.Embedding(vocab_size, output_dim)
create_dataloader_v1(raw_text,stride = max_length, max_length = max_length,
                     batch_size = 4, shuffle = False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[  40,  367, 2885, 1464]])

Inputs shape:
 torch.Size([1, 4])


In [152]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [242]:
#tokenized raw text
#sampled, indexed batches for loader
#len and getitems for abstract class
#Dataset can be any type of dataset, only inheritance and len and getitems 
# dudder matters,  init will look different for other dataset classes
class GPT_DatasetV1(Dataset):#Dataset Parameter?:
    def __init__(self, text,tokenizer, max_length, stride):
        
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text, allowed_special = {"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"
        #no assert, purpose of max_length, not batchsize, whole for loop, i
         #ndexing, out of bounds
        for i in range(0, len(token_ids) - max_length , stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self,idx):#return ids
        return self.input_ids[idx] , self.target_ids[idx]

In [243]:
#abstract, for reducing the repetition of rewriting tokenizer for dataset and dataset for when 
#for when when you use dataloader = DataLoader()

def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size = batch_size,
                            shuffle = shuffle, num_workers= num_workers,
                            drop_last = drop_last )
    return dataloader

In [244]:
torch.nn.Embedding?

Init signature:
torch.nn.Embedding(
    num_embeddings: int,
    embedding_dim: int,
    padding_idx: int | None = None,
    max_norm: float | None = None,
    norm_type: float = 2.0,
    scale_grad_by_freq: bool = False,
    sparse: bool = False,
    _weight: torch.Tensor | None = None,
    _freeze: bool = False,
    device=None,
    dtype=None,
) -> None
Docstring:     
A simple lookup table that stores embeddings of a fixed dictionary and size.

This module is often used to store word embeddings and retrieve them using indices.
The input to the module is a list of indices, and the output is the corresponding
word embeddings.

Args:
    num_embeddings (int): size of the dictionary of embeddings
    embedding_dim (int): the size of each embedding vector
    padding_idx (int, optional): If specified, the entries at :attr:`padding_idx` do not contribute to the gradient;
                                 therefore, the embedding vector at :attr:`padding_idx` is not updated during training

In [245]:
#lookup     #num_embeddings = vocab size, no of rows
output_dim = 256
vocab_size = 50257
token_embeddings_layer= torch.nn.Embedding(vocab_size,output_dim)

In [271]:
#positional
max_length=4
dataloader = create_dataloader_v1(raw_text,batch_size=8, max_length=4, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
positional_embedding_layer = torch.nn.Embedding(max_length,output_dim)


In [274]:
#culmination
print(token_embeddings_layer.weight.shape)
token_embeddings = token_embeddings_layer(inputs)
positional_embeddings = pos_embedding_layer(torch.arange(max_length))
print((inputs.shape))
print(positional_embeddings.shape)
print(token_embeddings.shape)

torch.Size([50257, 256])
torch.Size([8, 4])
torch.Size([4, 256])
torch.Size([8, 4, 256])


In [275]:
#embedding layerinput_embeddings = token_embeddings + positional_embeddingss
input_embeddings = token_embeddings +positional_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [279]:
torch.softmax?

Docstring:
softmax(input, dim, *, dtype=None) -> Tensor

Alias for :func:`torch.nn.functional.softmax`.
Type:      builtin_function_or_method